# 3. Calibrate the fitted vectors

Run **02_fitting.ipynb** first. This notebook reads fitted vectors and applies the original calibration function as an independent post-processing step.

The affine transform maps lattice 1 to a reference basis, then applies the same transform to lattice 2. It returns corrected vectors and twist angles; it does not resample the diffraction image or modify the fit.

Reference values are real-space **d spacings**. For sampling in `1/nm` per pixel, enter d spacings in `nm`. The original algorithm uses `sampling[0]`, including when two sampling values are stored.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

# Locate this checkout when launched from the project or an examples directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/moire_cdw").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
from moire_cdw.results import load_fit_result
from moire_cdw.calibration import calibration_from_vectors

## Reference settings

The values below reproduce the old calibration example (`0.276 nm`, `60 degrees`); they are not independently established for another material or reflection. Change them to the reference appropriate for your selected lattice 1.

In [ ]:
FIT_FILE = ROOT / "outputs/two_lattice/fit.npz"
OUTPUT = ROOT / "outputs/two_lattice/calibration.npz"
D_U1 = 0.276
D_V1 = 0.276
REFERENCE_ANGLE_DEG = 60.0

## Read the fitting result

In [ ]:
if not FIT_FILE.exists():
    raise FileNotFoundError("Run 02_fitting.ipynb first.")
result = load_fit_result(FIT_FILE)
summary = result["summary"]
vectors = {key: summary[key] for key in ("origin_rc", "u1_rc", "v1_rc", "u2_rc", "v2_rc")}
print("Input:", result["metadata"]["source_file"])
print("Sampling:", result["sampling"], "Units:", result["units"])
print("Fit settings:", result["metadata"]["fit_settings"])
if result["units"] != ["1/nm", "1/nm"]:
    raise ValueError("This example assumes d spacings in nm and sampling in 1/nm. Check units before calibration.")

## Apply calibration to the fitted vectors

In [ ]:
info = calibration_from_vectors(
    **vectors, sampling=result["sampling"], u1_calibrate=D_U1, v1_calibrate=D_V1,
    angle1_calibrate=REFERENCE_ANGLE_DEG, return_info=True, plot_vectors=True,
)
print("Twist before calibration:", info["twist_angle_before"])
print("Twist after calibration:", info["twist_angle_after"])

## Save calibration separately

The fitting result remains available unchanged. `A` operates on `(row, col)` vectors in binned pixels. Corrected reciprocal lengths are obtained by multiplying their norms by the original first sampling value.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(OUTPUT, **info, **vectors, sampling=result["sampling"],
                    units=np.asarray(result["units"]), source_fit=str(FIT_FILE),
                    d_u1=D_U1, d_v1=D_V1, reference_angle_deg=REFERENCE_ANGLE_DEG)
print("Saved:", OUTPUT)

For the original peak-index interface, use `from moire_cdw.calibration import calibration`, then call `calibration(peaks, vector_indices, im_bin, ...)`. Its defaults and return behavior are unchanged.